In [40]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv



In [41]:
load_dotenv()

modal = ChatGoogleGenerativeAI(
     model="gemini-3.5-flash-lite",
)

In [42]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [43]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    
    prompt = f'Generate a detailed outline for a blog on a topic - {title}'
    outline = modal.invoke(prompt).content
    
    state['outline'] = outline
    
    return state

In [44]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    
    prompt = f'Generate a detailed blog on the title - {title} using the following outline \n {outline}'
    content = modal.invoke(prompt).content
    
    state['content'] = content
    
    return state

In [45]:
graph = StateGraph(BlogState)


graph.add_node("create_outline" , create_outline)
graph.add_node("create_blog" , create_blog)


graph.add_edge(START , "create_outline")
graph.add_edge("create_outline" , "create_blog")
graph.add_edge("create_blog" , END)

workflow = graph.compile()

In [46]:
initial_state = {"title": "Rise of AI in India."}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India.', 'outline': [{'type': 'text', 'text': 'Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in India."** \n\n---\n\n# Blog Title Options:\n*   *From Code to Cognition: The Meteoric Rise of AI in India*\n*   *Bharat 2.0: How Artificial Intelligence is Reshaping India’s Future*\n*   *The AI Boom in India: Opportunities, Challenges, and the Road Ahead*\n\n---\n\n## I. Introduction\n*   **The Hook:** Start with a compelling stat or trend (e.g., India’s rapid digital transformation, high smartphone penetration, or massive tech talent pool).\n*   **The Context:** Briefly define the current AI boom globally and how India is transitioning from being a back-office IT hub to an AI innovation powerhouse.\n*   **Thesis Statement:** India’s rise as an AI superpower is driven by government initiatives, a massive developer ecosystem, and innovative startups solving grassroots problems.\n\n---\n\n## II. The Catalyst: Why Now? (Dri

In [49]:
print(final_state['outline'])

[{'type': 'text', 'text': 'Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in India."** \n\n---\n\n# Blog Title Options:\n*   *From Code to Cognition: The Meteoric Rise of AI in India*\n*   *Bharat 2.0: How Artificial Intelligence is Reshaping India’s Future*\n*   *The AI Boom in India: Opportunities, Challenges, and the Road Ahead*\n\n---\n\n## I. Introduction\n*   **The Hook:** Start with a compelling stat or trend (e.g., India’s rapid digital transformation, high smartphone penetration, or massive tech talent pool).\n*   **The Context:** Briefly define the current AI boom globally and how India is transitioning from being a back-office IT hub to an AI innovation powerhouse.\n*   **Thesis Statement:** India’s rise as an AI superpower is driven by government initiatives, a massive developer ecosystem, and innovative startups solving grassroots problems.\n\n---\n\n## II. The Catalyst: Why Now? (Drivers of India’s AI Boom)\n*   **A. Massive Di